# Objective
Here, we'll perform the normality tests.

In [ ]:
library(here)
library(ggpubr)
library(maplet)
library(dplyr)
library(purrr)

# set repo path
repo <- here()
renv::activate(project = repo)

In [ ]:
# load maplet object
D <- readRDS(here('data', 'preprocessed_venous_metabolon.RDS'))

# Normality Tests

## Shapiro Test

In [ ]:
# extract assay data
mets <- D %>% assay() %>% t() %>% as.data.frame()

normality_results <- mets %>%
  tidyr::pivot_longer(
    cols = everything(),
    names_to = "metabolite",
    values_to = "value"
  ) %>%
  group_by(metabolite) %>%
  summarise(
    n = sum(!is.na(value)),
    mean = mean(value, na.rm = TRUE),
    sd = sd(value, na.rm = TRUE),
    # run shapiro test only if n is between 3 and 5000 and sd is greater than 0 (non-constant)
    shapiro_p = if (n >= 3 && n <= 5000 && sd(value, na.rm = TRUE) > 0) shapiro.test(value)$p.value else NA_real_,
    .groups = "drop"
  ) %>%
  mutate(
    p_adjusted = p.adjust(shapiro_p, method = "bonferroni"),
    normal_at_0.05 = shapiro_p > 0.05
  ) %>%
  arrange(shapiro_p)

In [ ]:
normality_results %>% head()

In [ ]:
options(repr.plot.width = 4, repr.plot.height = 3)

gghistogram(normality_results, x = "shapiro_p", bins = 30, fill = "steelblue") +
 labs(title = "Shapiro-Wilk p-values")

normality_results %>% dplyr::count(normal_at_0.05)

## Histograms

In [ ]:
# pdf(here::here('outputs', 'histograms_all_metabolites.pdf'), width = 4, height = 4)

# for (j in seq_along(mets)) {
#   x <- mets[[j]]
#   x <- x[!is.na(x)]

#   if (length(x) >= 3 && stats::sd(x) > 0) {
#     print(
#       ggpubr::gghistogram(
#         data.frame(x = x),
#         x = "x",
#         bins = 30,
#         fill = "steelblue"
#       ) +
#         ggplot2::labs(
#           title = names(mets)[j], 
#           x = "", 
#           y = ""
#         ) +
#         ggplot2::theme(
#           plot.title = ggplot2::element_text(size = 8),
#           axis.text = ggplot2::element_text(size = 6)
#         )
#     )
#   } else {
#     plot.new()
#     title(main = names(mets)[j], cex.main = 0.5)
#     text(0.5, 0.5, "Insufficient\nvariation", cex = 0.5)
#   }
# }

# dev.off()

## QQ Plot

In [ ]:
# pdf(here::here('outputs', 'qqplots_all_metabolites.pdf'), width = 4, height = 4)

# par(
#   mfrow = c(5, 5),
#   mar = c(1.2, 1.2, 1.8, 0.5),
#   mgp = c(1, 0.2, 0),
#   tcl = -0.2,
#   cex.axis = 0.4,
#   cex.main = 0.5
# )

# for (j in seq_along(mets)) {
#   x <- mets[[j]]
#   x <- x[!is.na(x)]

#   if (length(x) >= 3 && stats::sd(x) > 0) {
#     qqnorm(x,
#       main = names(mets)[j],
#       xlab = "",
#       ylab = "",
#       pch = 16,
#       cex = 0.25
#     )
#     qqline(x)
#   } else {
#     plot.new()
#     title(main = names(mets)[j], cex.main = 0.5)
#     text(0.5, 0.5, "Insufficient\nvariation", cex = 0.5)
#   }
# }

# dev.off()

# LM Residuals

In [ ]:
plot_met_residual_qq <- function(maplet_obj,
                                 var_of_interest,
                                 met_names,
                                 display_names = NULL,
                                 ncol = 4,
                                 nrow = 4,
                                 point_size = 0.8,
                                 title_size = 8,
                                 missing_label = "Model\nnot found",
                                 insufficient_label = "Insufficient\nvariation") {

  if (!is.null(display_names) && length(display_names) != length(met_names))
    stop("`display_names` must be the same length as `met_names`.")

  res <- maplet::mtm_get_stat_by_name(
    maplet_obj,
    paste(var_of_interest, "met"),
    fullstruct = TRUE
  )

  mods <- res$lstobj

  plots <- vector("list", length(met_names))
  names(plots) <- met_names

  for (i in seq_along(met_names)) {
    met_name    <- met_names[[i]]
    plot_title  <- if (!is.null(display_names)) display_names[[i]] else met_name

    if (met_name %in% names(mods)) {
      r <- residuals(mods[[met_name]])
      r <- r[!is.na(r)]

      if (length(r) >= 3 && stats::sd(r) > 0) {
        df <- data.frame(sample = r)
        p <- ggplot2::ggplot(df, ggplot2::aes(sample = sample)) +
          ggplot2::stat_qq(size = point_size, shape = 16) +
          ggplot2::stat_qq_line() +
          ggplot2::labs(title = plot_title, x = NULL, y = NULL) +
          ggplot2::theme_bw(base_size = title_size) +
          ggplot2::theme(
            plot.title = ggplot2::element_text(size = title_size, hjust = 0.5)
          )
      } else {
        p <- ggplot2::ggplot() +
          ggplot2::annotate(
            "text", x = 0.5, y = 0.5, label = insufficient_label,
            size = title_size / 2.845, hjust = 0.5, vjust = 0.5
          ) +
          ggplot2::labs(title = plot_title) +
          ggplot2::theme_void() +
          ggplot2::theme(
            plot.title = ggplot2::element_text(size = title_size, hjust = 0.5)
          )
      }

    } else {
      p <- ggplot2::ggplot() +
        ggplot2::annotate(
          "text", x = 0.5, y = 0.5, label = missing_label,
          size = title_size / 2.845, hjust = 0.5, vjust = 0.5
        ) +
        ggplot2::labs(title = plot_title) +
        ggplot2::theme_void() +
        ggplot2::theme(
          plot.title = ggplot2::element_text(size = title_size, hjust = 0.5)
        )
    }

    plots[[met_name]] <- p
  }

  n_per_page <- ncol * nrow
  n_pages <- ceiling(length(plots) / n_per_page)

  for (page in seq_len(n_pages)) {
    idx <- seq((page - 1) * n_per_page + 1, min(page * n_per_page, length(plots)))
    grid <- patchwork::wrap_plots(plots[idx], ncol = ncol, nrow = nrow)
    print(grid)
  }

  invisible(mods)
}

In [ ]:
plot_met_residual_fitted <- function(maplet_obj,
                                     var_of_interest,
                                     met_names,
                                     display_names = NULL,
                                     ncol = 4,
                                     nrow = 4,
                                     point_size = 0.8,
                                     title_size = 8,
                                     missing_label = "Model\nnot found",
                                     insufficient_label = "No usable\nresiduals") {

  if (!is.null(display_names) && length(display_names) != length(met_names))
    stop("`display_names` must be the same length as `met_names`.")

  res <- maplet::mtm_get_stat_by_name(
    maplet_obj,
    paste(var_of_interest, "met"),
    fullstruct = TRUE
  )

  mods <- res$lstobj

  plots <- vector("list", length(met_names))
  names(plots) <- met_names

  for (i in seq_along(met_names)) {
    met_name   <- met_names[[i]]
    plot_title <- if (!is.null(display_names)) display_names[[i]] else met_name

    if (met_name %in% names(mods)) {
      fit <- mods[[met_name]]
      r   <- resid(fit)
      f   <- fitted(fit)

      keep <- !is.na(r) & !is.na(f)
      r <- r[keep]
      f <- f[keep]

      if (length(r) >= 3 && stats::sd(r) > 0) {
        df <- data.frame(fitted = f, residuals = r)
        p <- ggplot2::ggplot(df, ggplot2::aes(x = fitted, y = residuals)) +
          ggplot2::geom_point(size = point_size, shape = 16) +
          ggplot2::geom_hline(yintercept = 0, linetype = "dashed") +
          ggplot2::labs(title = plot_title, x = "Fitted values", y = "Residuals") +
          ggplot2::theme_bw(base_size = title_size) +
          ggplot2::theme(
            plot.title = ggplot2::element_text(size = title_size, hjust = 0.5)
          )
      } else {
        p <- ggplot2::ggplot() +
          ggplot2::annotate(
            "text", x = 0.5, y = 0.5, label = insufficient_label,
            size = title_size / 2.845, hjust = 0.5, vjust = 0.5
          ) +
          ggplot2::labs(title = plot_title) +
          ggplot2::theme_void() +
          ggplot2::theme(
            plot.title = ggplot2::element_text(size = title_size, hjust = 0.5)
          )
      }

    } else {
      p <- ggplot2::ggplot() +
        ggplot2::annotate(
          "text", x = 0.5, y = 0.5, label = missing_label,
          size = title_size / 2.845, hjust = 0.5, vjust = 0.5
        ) +
        ggplot2::labs(title = plot_title) +
        ggplot2::theme_void() +
        ggplot2::theme(
          plot.title = ggplot2::element_text(size = title_size, hjust = 0.5)
        )
    }

    plots[[met_name]] <- p
  }

  n_per_page <- ncol * nrow
  n_pages <- ceiling(length(plots) / n_per_page)

  for (page in seq_len(n_pages)) {
    idx <- seq((page - 1) * n_per_page + 1, min(page * n_per_page, length(plots)))
    grid <- patchwork::wrap_plots(plots[idx], ncol = ncol, nrow = nrow)
    print(grid)
  }

  invisible(mods)
}

In [ ]:
# extract metabolite names
met_names <- rownames(D)

In [ ]:
# Define confounders as a string
confounders <- "age + SEX + bmi + f_wnowt"

## Re-run LM

In [ ]:
# specify variable of interest 
var_of_interest <- "RVGLOB6n"

# Construct the lm formula
full_formula <- as.formula(paste("~", var_of_interest, "+", confounders))

# run univariate tests
D_GLS <- D %>%
  # Linear model with dynamically created formula
  mt_stats_univ_lm(formula = full_formula, stat_name = paste(var_of_interest, "met")) %>%
  # Add multiple testing correction
  mt_post_multtest(stat_name = paste(var_of_interest, "met"), method = "fdr") %>%
  # Add stats logging
  mt_reporting_stats(stat_name = paste(var_of_interest, "met"), stat_filter = p.adj < 0.05) %>% 
  {.}

In [ ]:
# specify variable of interest 
var_of_interest <- "RVFACn"

# Construct the lm formula
full_formula <- as.formula(paste("~", var_of_interest, "+", confounders))

# run univariate tests
D_FAC <- D %>%
  # Linear model with dynamically created formula
  mt_stats_univ_lm(formula = full_formula, stat_name = paste(var_of_interest, "met")) %>%
  # Add multiple testing correction
  mt_post_multtest(stat_name = paste(var_of_interest, "met"), method = "fdr") %>%
  # Add stats logging
  mt_reporting_stats(stat_name = paste(var_of_interest, "met"), stat_filter = p.adj < 0.05) %>% 
  {.}

In [ ]:
# specify variable of interest 
var_of_interest <- "RVFACn"

# Construct the lm formula
full_formula <- as.formula(paste("~", var_of_interest, "+", confounders))

# run univariate tests
D2 <- D %>%
  # Linear model with dynamically created formula
  mt_stats_univ_lm(formula = full_formula, stat_name = paste(var_of_interest, "met")) %>%
  # Add multiple testing correction
  mt_post_multtest(stat_name = paste(var_of_interest, "met"), method = "fdr") %>%
  # Add stats logging
  mt_reporting_stats(stat_name = paste(var_of_interest, "met"), stat_filter = p.adj < 0.05) %>% 
  {.}

In [ ]:
# specify variable of interest 
var_of_interest <- "mri_RVEF"

# Construct the lm formula
full_formula <- as.formula(paste("~", var_of_interest, "+", confounders))

# run univariate tests
D_RVEF <- D %>%
  # Linear model with dynamically created formula
  mt_stats_univ_lm(formula = full_formula, stat_name = paste(var_of_interest, "met")) %>%
  # Add multiple testing correction
  mt_post_multtest(stat_name = paste(var_of_interest, "met"), method = "fdr") %>%
  # Add stats logging
  mt_reporting_stats(stat_name = paste(var_of_interest, "met"), stat_filter = p.adj < 0.05) %>% 
  {.}

## GLS

In [ ]:
# extract formal/display names from rowData (parallel to met_names / rownames)
met_formal_names <- D_GLS %>% rowData() %>% as.data.frame() %>% pull(name)

In [ ]:
pdf(here::here("outputs", "qqplots_residuals_all_metabolites_GLS.pdf"),
    width = 8, height = 9)

plot_met_residual_qq(
  maplet_obj = D_GLS,
  var_of_interest = "RVGLOB6n",
  met_names = met_names,
  display_names = met_formal_names
)

dev.off()

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 4)

# specify metabolite 
metabolite <- 'X1.5.anhydroglucitol..1.5.AG.'

## extract the residuals for a single metabolite: 
res <- maplet::mtm_get_stat_by_name(D_GLS, paste("RVGLOB6n", "met"), fullstruct = TRUE) 
mods <- res$lstobj 
r <- residuals(mods[[metabolite]])

# plot histogram of metabolite
met <- D %>% assay() %>% t() %>% as.data.frame() %>% pull(metabolite)

# plot histogram of residuals and metabolites
par(mfrow = c(1, 2))
hist(met, main = metabolite, xlab = "", ylab = "", breaks = 30)
hist(r, main = paste("Residuals of", metabolite), xlab = "", ylab = "", breaks = 30)

In [ ]:
pdf(here::here("outputs", "residuals_vs_fitted_all_metabolites_GLS.pdf"),
  width = 8, height = 9)

plot_met_residual_fitted(
  maplet_obj = D_GLS,
  var_of_interest = "RVGLOB6n",
  met_names = met_names,
  display_names = met_formal_names
)

dev.off()

## FAC

In [ ]:
pdf(here::here("outputs", "qqplots_residuals_all_metabolites_FAC.pdf"),
    width = 8, height = 9)

plot_met_residual_qq(
  maplet_obj = D_FAC,
  var_of_interest = "RVFACn",
  met_names = met_names,
  display_names = met_formal_names
)

dev.off()

In [ ]:
pdf(here::here("outputs", "residuals_vs_fitted_all_metabolites_FAC.pdf"),
    width = 8, height = 9)

plot_met_residual_fitted(
  maplet_obj = D_FAC,
  var_of_interest = "RVFACn",
  met_names = met_names,
  display_names = met_formal_names
)

dev.off()

## RVEF

In [ ]:
pdf(here::here("outputs", "qqplots_residuals_all_metabolites_RVEF.pdf"),
    width = 8, height = 9)

plot_met_residual_qq(
  maplet_obj = D_RVEF,
  var_of_interest = "mri_RVEF",
  met_names = met_names,
  display_names = met_formal_names
)

dev.off()

In [ ]:
pdf(here::here("outputs", "residuals_vs_fitted_all_metabolites_RVEF.pdf"),
  width = 8, height = 9)

plot_met_residual_fitted(
  maplet_obj = D_RVEF,
  var_of_interest = "mri_RVEF",
  met_names = met_names,
  display_names = met_formal_names
)

dev.off()

## Subset on 'Top Hits'

In [ ]:
top_hits <- c('histidine', 'vanillylmandelate..VMA.', 'androstenediol..3beta.17beta..monosulfate..1.',
     'cholesterol',  'vanillactate', 'dehydroepiandrosterone.sulfate..DHEA.S.', 'homoarginine', 'X3.hydroxy.3.methylglutarate',
     'threonine',  'N.carbamoylvaline')

### GLS

In [ ]:
pdf(here::here("outputs", "qqplots_residuals_top_hits_GLS.pdf"),
    width = 4, height = 4.5)

plot_met_residual_qq(
  maplet_obj = D_GLS,
  var_of_interest = "RVGLOB6n",
  met_names = top_hits
)

dev.off()

In [ ]:
pdf(here::here("outputs", "residuals_vs_fitted_top_hits_GLS.pdf"),
    width = 4, height = 4.5)

plot_met_residual_fitted(
  maplet_obj = D_GLS,
  var_of_interest = "RVGLOB6n",
  met_names = top_hits
)

dev.off()

### FAC

In [ ]:
pdf(here::here("outputs", "qqplots_residuals_top_hits_FAC.pdf"),
    width = 4, height = 4.5)

plot_met_residual_qq(
  maplet_obj = D_FAC,
  var_of_interest = "RVFACn",
  met_names = top_hits
)

dev.off()

In [ ]:
pdf(here::here("outputs", "residuals_vs_fitted_top_hits_FAC.pdf"),
    width = 4, height = 4.5)

plot_met_residual_fitted(
  maplet_obj = D_FAC,
  var_of_interest = "RVFACn",
  met_names = top_hits
)

dev.off()

### RVEF

In [ ]:
pdf(here::here("outputs", "qqplots_residuals_top_hits_RVEF.pdf"),
    width = 4, height = 4.5)

plot_met_residual_qq(
  maplet_obj = D_RVEF,
  var_of_interest = "mri_RVEF",
  met_names = top_hits
)

dev.off()

In [ ]:
pdf(here::here("outputs", "residuals_vs_fitted_top_hits_RVEF.pdf"),
    width = 4, height = 4.5)

plot_met_residual_fitted(
  maplet_obj = D_RVEF,
  var_of_interest = "mri_RVEF",
  met_names = top_hits
)

dev.off()

# Single Metabolite Tester


In [ ]:
options(repr.plot.width = 8, repr.plot.height = 4)

# specify metabolite 
metabolite <- 'fructose'

## extract the residuals for a single metabolite: 
res <- maplet::mtm_get_stat_by_name(D_GLS, paste("RVGLOB6n", "met"), fullstruct = TRUE) 
mods <- res$lstobj 
r <- residuals(mods[[metabolite]])

# plot histogram of metabolite
met <- D %>% assay() %>% t() %>% as.data.frame() %>% pull(metabolite)

# plot histogram of residuals and metabolites with normal distributions
par(mfrow = c(1, 2))

# Histogram of metabolite with theoretical normal distribution overlay
hist(met, main = metabolite, xlab = "", ylab = "", breaks = 30, freq = FALSE)
met_mean <- mean(met, na.rm = TRUE)
met_sd <- sd(met, na.rm = TRUE)
# curve(dnorm(x, mean = met_mean, sd = met_sd), 
    #   col = "red", lwd = 2, add = TRUE)

# Histogram of residuals with theoretical normal distribution overlay
hist(r, main = paste("Residuals of", metabolite), xlab = "", ylab = "", breaks = 30, freq = FALSE)
r_mean <- mean(r, na.rm = TRUE)
r_sd <- sd(r, na.rm = TRUE)
    # curve(dnorm(x, mean = r_mean, sd = r_sd), 
    #   col = "red", lwd = 2, add = TRUE)